# Live RL — training **fox** only (critic-free PPO)

Single-species version of `train.ipynb`. Only the **fox** keep learning as they live: while
awake they collect on-policy experience, and when the fox population beds down at night the
simulation **pauses** and one PPO update runs on the fox policy alone. On waking, collection
resumes. The **sheep** are *not* learning here — they are a fixed opponent that keeps the
ecosystem realistic (predator–prey pressure) while the fox policy adapts against a
**stationary** environment (which is what makes single-species RL stable).

## The non-training species (sheep) — pick one of two drivers

Set `OPPONENT` in the config cell:

- `"rule"` — the calibrated `RuleBrain` (see v1.md §18 / CLAUDE.md). This is the well-tuned
  teacher that makes predator–prey coexist, so the ecosystem stays in its known-good regime.
  **Recommended default** — a persistent, stationary opponent.
- `"model"` — an already-trained TorchScript policy for the sheep (the imitation clone
  `../imitation_learning/sheep.pt`, or a `sheep_ppo.pt` you trained earlier). Loaded frozen via
  `sim.policy_brain` and never updated. Use this to fine-tune fox against a *neural* sheep.
  ⚠️ the raw imitation sheep clone can be fragile (the cloned fox can die out through prey
  troughs) — if the sheep collapses, the fox train against an unrealistic world; prefer
  `"rule"` unless you have a robust sheep checkpoint.

**Reward / pain** and **critic-free PPO** are exactly as in `train.ipynb` (all machinery lives
in `ppo_live.py`); only the fox policy is optimised. The result re-exports as a drop-in
`fox_ppo.pt` TorchScript archive for `run_experiment.py` / `run_live.py`.

In [ ]:
# --- bootstrap: reach ppo_live.py (this folder) + the imitation toolkit + the repo root ---
import sys, time
from pathlib import Path

_HERE = Path.cwd()
for _c in (_HERE, *_HERE.parents):
    if (_c / "notebooks" / "live_learning" / "ppo_live.py").exists():
        _LIVE = _c / "notebooks" / "live_learning"
        break
    if (_c / "ppo_live.py").exists():
        _LIVE = _c
        break
else:
    raise RuntimeError("run this notebook from inside the ecosystem repo")
if str(_LIVE) not in sys.path:
    sys.path.insert(0, str(_LIVE))
_NB = _LIVE.parent                            # notebooks/ -- holds the shared common.py
if str(_NB) not in sys.path:
    sys.path.insert(0, str(_NB))

import numpy as np
import torch
import matplotlib.pyplot as plt

import ppo_live as P            # the PPO engine (imports common.py -> puts repo root on sys.path)
import common as C
from config import make_config, SHEEP, FOX, SPECIES_NAMES
from sim.simulation import Simulation
from sim.policy_brain import policy_brain_from_path   # frozen TorchScript opponent

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# --- configuration ---------------------------------------------------------------------
TRAIN  = FOX          # the species that LEARNS in this notebook
OTHER  = SHEEP          # the fixed (non-learning) opponent species

WORLD_SEED = 12345         # fixes the map (terrain + hydrology)
RUN_SEED   = 7             # fixes the sim dynamics RNG (action sampling adds its own noise)
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

# ---- how the NON-training species (sheep) is driven: "rule" or "model" ----
OPPONENT      = "model"                       # "rule" -> calibrated RuleBrain (recommended)
                                             # "model" -> a frozen trained policy (path below)
OPPONENT_PATH = C.MODEL_PATHS[OTHER]         # used only when OPPONENT == "model"
                                             # (imitation clone; or set to a *_ppo.pt you trained)

# how long to train. One "cycle" == one day of collection + one night PPO update.
N_CYCLES   = 40            # raise for a longer run; each cycle is ~1 in-game day (~240 ticks)
MAX_TICKS  = 200_000       # hard safety cap on total sim ticks
SAVE_EVERY = 5             # export a deployable checkpoint every N cycles

# warm start the TRAIN species from its imitation clone (set to None to learn from scratch)
WARM_TRAIN = C.MODEL_PATHS[TRAIN]

# where the fine-tuned, deployable fox policy is written (kept separate from the clone)
OUT_TRAIN = _LIVE / f"{SPECIES_NAMES[TRAIN]}_ppo.pt"

ppocfg = P.PPOConfig(
    gamma=0.99, clip=0.2, epochs=4, minibatch=1024, lr=1e-4,
    entropy_coef=0.005, max_grad_norm=0.5, target_kl=0.03,
    max_agents=128,        # trajectories tracked for the train species per day (bounds memory)
    night_hi=0.5,          # >= this fraction of fox asleep  -> pause + train
    night_lo=0.2,          # <= this fraction of fox asleep  -> daytime, resume collecting
    min_cycle=60,          # min ticks collected before a night trigger is allowed
    horizon=400,           # force a train if a day runs longer than this (safety)
)
rcfg = P.RewardConfig()    # reward/pain weights -- tweak freely

torch.manual_seed(RUN_SEED)   # reproducible action sampling (independent of the sim's numpy RNG)
assert OPPONENT in ("rule", "model")
print(f"training {SPECIES_NAMES[TRAIN]} | {SPECIES_NAMES[OTHER]} driven by: {OPPONENT}")
ppocfg, rcfg

In [ ]:
# --- build the ONE learning policy + the fixed opponent, wired into a per-species sim -----
# the learner: a LivePPOBrain holding ONLY the train species' policy (warm-started).
policy = P.build_ppo_policy(TRAIN, warm_start=WARM_TRAIN, device=DEVICE)
optimizer = torch.optim.Adam(policy.parameters(), lr=ppocfg.lr)
buffer = P.RolloutBuffer(gamma=ppocfg.gamma, max_agents=ppocfg.max_agents)

brain = P.LivePPOBrain({TRAIN: policy}, device=DEVICE)   # drives the TRAIN species only
brain.training = True

# the opponent driver for the OTHER species:
#   "rule"  -> pass None, so Simulation fills it with a shared RuleBrain on the run RNG.
#   "model" -> a frozen PolicyBrain loaded from a TorchScript checkpoint (never updated).
if OPPONENT == "model":
    opponent = policy_brain_from_path(str(OPPONENT_PATH), OTHER, device=DEVICE)
    print("opponent:", OPPONENT_PATH)
else:
    opponent = None                                        # -> calibrated RuleBrain
    print("opponent: calibrated RuleBrain")

cfg = make_config(world_seed=WORLD_SEED, seed=RUN_SEED)
# per-species brain spec: the learner on TRAIN, the fixed opponent on OTHER. Simulation wraps
# this in a CompositeBrain that routes each species to its own driver.
sim = Simulation(cfg, brain={TRAIN: brain, OTHER: opponent})
ent = sim.entities
print("seeded:", sim.populations)

## The live training loop

Identical state machine to the combined notebook, but the day/night trigger follows the
**fox**'s own sleep (the species we collect from):

- **Not collecting** → wait for fox daybreak (`asleep_frac ≤ night_lo`), then start a day of
  collection.
- **Collecting** → each tick snapshot state → `sim.step()` (the fox brain samples + records) →
  reward by snapshot-diff → file into the fox buffer. When most fox bed down
  (`asleep_frac ≥ night_hi`, after ≥ `min_cycle` ticks) — or the `horizon` safety — **pause and
  run one PPO update** on the fox policy, clear the buffer, sleep through the night.

The sheep act every tick from their fixed brain but are never recorded or updated. Sleeping /
dusk-straggler fox whose action the sleep system overrides are dropped via the `controlled`
mask, exactly as before.

In [ ]:
# --- run it -------------------------------------------------------------------------------
history = []      # per-cycle metrics
pop_log = []      # (tick, n_sheep, n_fox) every tick, for the population plot

collecting = False
cycle = 0
cycle_start = sim.tick
t0 = time.time()

while cycle < N_CYCLES and sim.tick < MAX_TICKS:
    snap = P.snapshot(ent)
    brain.collecting = collecting
    stats = sim.step()
    pop_log.append((sim.tick, stats["n_sheep"], stats["n_fox"]))

    # stop if the TRAIN species dies out (no experience to learn from) or the world empties
    n_train = ent.count_species(TRAIN)
    if n_train == 0 or ent.n_alive == 0:
        print(f"{SPECIES_NAMES[TRAIN]} went extinct — stopping")
        break

    # reward every TRAIN agent that acted this tick and file it into the buffer
    for sid, pend in brain.pending.items():          # pending holds only the TRAIN species
        r, done, controlled = P.compute_rewards(rcfg, snap, ent, pend)
        buffer.add(pend, r, done, controlled)

    # night detection follows the TRAIN species' own sleep (that's whose day we collect)
    tmask = ent.species_mask(TRAIN)
    n_asleep = int(ent.asleep[tmask].sum())
    asleep_frac = n_asleep / max(1, n_train)

    if not collecting:
        if asleep_frac <= ppocfg.night_lo:      # daybreak -> begin a new day of collection
            collecting = True
            cycle_start = sim.tick
    else:
        elapsed = sim.tick - cycle_start
        if elapsed >= ppocfg.min_cycle and (asleep_frac >= ppocfg.night_hi
                                            or elapsed >= ppocfg.horizon):
            # ---- NIGHT: pause the sim and run one PPO update on the TRAIN policy ----
            nm = SPECIES_NAMES[TRAIN]
            rec = {"cycle": cycle, "tick": sim.tick, "day_len": elapsed,
                   "n_sheep": stats["n_sheep"], "n_fox": stats["n_fox"]}
            msg = [f"cycle {cycle:3d} | tick {sim.tick:6d} | day {elapsed:3d}t | "
                   f"sheep {stats['n_sheep']:4d} fox {stats['n_fox']:3d}"]
            nt = buffer.n_transitions()
            batch = buffer.build_batch()
            if batch is None:
                msg.append(f"    {nm:5s}: no data collected")
            else:
                m = P.ppo_update(policy, optimizer, batch, ppocfg, device=DEVICE)
                rec[f"{nm}_reward"] = float(batch["returns"].mean())
                rec[f"{nm}_ploss"] = m["policy_loss"]
                rec[f"{nm}_entropy"] = m["entropy"]
                rec[f"{nm}_kl"] = m["approx_kl"]
                rec[f"{nm}_n"] = nt
                msg.append(f"    {nm:5s}: n={nt:6d} skip={buffer.skipped:6d} "
                           f"meanR={batch['returns'].mean():+.3f} "
                           f"ploss={m['policy_loss']:+.4f} ent={m['entropy']:.3f} "
                           f"kl={m['approx_kl']:.4f} clip={m['clipfrac']:.3f}")
            buffer.clear()
            history.append(rec)
            print("\n".join(msg))

            if (cycle + 1) % SAVE_EVERY == 0:
                P.export_policy(TRAIN, policy, OUT_TRAIN, meta={"cycle": cycle})
                print(f"    [saved checkpoint @ cycle {cycle}]")

            collecting = False
            cycle += 1

print(f"\ndone: {cycle} cycles, {sim.tick} ticks, {time.time()-t0:.1f}s | final {sim.populations}")

In [ ]:
# --- export the final deployable fox policy (drop-in TorchScript archive) ----------------
P.export_policy(TRAIN, policy, OUT_TRAIN, meta={"cycles": cycle, "world_seed": WORLD_SEED})
print("wrote", OUT_TRAIN)

_other_flag = (f"--{SPECIES_NAMES[OTHER]}-brain " + str(OPPONENT_PATH)) if OPPONENT == "model" else ""
print("\nDeploy with, e.g.:")
print(f"  venv/Scripts/python.exe run_experiment.py --ticks 3000 --world-seed {WORLD_SEED} "
      f"--seed {RUN_SEED} \\\n    --{SPECIES_NAMES[TRAIN]}-brain {OUT_TRAIN} {_other_flag}"
      f"--out runs/{SPECIES_NAMES[TRAIN]}_ppo_eval.csv --plot")

## Metrics

In [ ]:
# --- learning curve (train species) + population dynamics ---------------------------------
if history:
    nm = SPECIES_NAMES[TRAIN]
    rk = f"{nm}_reward"; ek = f"{nm}_entropy"; kk = f"{nm}_kl"
    fig, ax = plt.subplots(2, 2, figsize=(13, 8))

    xs = [h["cycle"] for h in history if rk in h]
    ax[0, 0].plot(xs, [h[rk] for h in history if rk in h], label=nm)
    ax[0, 1].plot(xs, [h[ek] for h in history if ek in h], label=nm)
    ax[1, 1].plot(xs, [h[kk] for h in history if kk in h], label=nm)
    ax[0, 0].set(title="mean discounted return / cycle", xlabel="cycle", ylabel="return")
    ax[0, 0].legend(); ax[0, 0].grid(alpha=.3)
    ax[0, 1].set(title="policy entropy (exploration)", xlabel="cycle"); ax[0, 1].legend(); ax[0, 1].grid(alpha=.3)
    ax[1, 1].set(title="approx KL / cycle (trust region)", xlabel="cycle"); ax[1, 1].legend(); ax[1, 1].grid(alpha=.3)

    if pop_log:
        pl = np.array(pop_log)
        ax[1, 0].plot(pl[:, 0], pl[:, 1], label="sheep")
        ax[1, 0].plot(pl[:, 0], pl[:, 2], label="fox")
        ax[1, 0].set(title="population over time", xlabel="tick", ylabel="count")
        ax[1, 0].legend(); ax[1, 0].grid(alpha=.3)
    plt.tight_layout(); plt.show()
else:
    print("no cycles completed yet — run the training loop above")